# NILM-Engine — Run 4: 전체 데이터 단일 학습 (EXP_ALL)

| 항목 | 값 |
|------|----|
| 실험 전략 | 4주 누적 없이 scratch → 전체 기간 1회 학습 |
| 모델 | cnn_tda |
| Train | house_011,015,016,017,033,039,054,063 (전체 기간) |
| Val | house_049 (전체 기간) |
| Test | house_067 (전체 기간) |
| 목적 | EXP3/4 val MAE 악화 패턴이 누적 전략 탓인지 아키텍처 한계인지 구분 |

---

## 1. 환경 설치 & GCS 인증

In [ ]:
!pip install -q gudhi

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("GCS 인증 완료")

## 2. GCS 연결 검증

In [ ]:
import gcsfs
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from pyarrow.fs import PyFileSystem, FSSpecHandler

gcs = gcsfs.GCSFileSystem()
_gcs_pa = PyFileSystem(FSSpecHandler(gcs))

print("=== 원천데이터 스키마 ===")
raw_ds = ds.dataset(
    "ax-nilm-data-dhwang0803-us/nilm/training_dev10",
    filesystem=_gcs_pa, partitioning=["household_id", "channel", "date"],
)
print(raw_ds.schema)

## 3. 분할 & 경로 상수

In [ ]:
SPLIT = {
    "train": ["house_011", "house_015", "house_016", "house_017",
              "house_033", "house_039", "house_054", "house_063"],
    "val":   ["house_049"],
    "test":  ["house_067"],
}
BUCKET_PREFIX = "ax-nilm-data-dhwang0803-us/nilm/training_dev10"
LABEL_PATH    = "ax-nilm-data-dhwang0803-us/nilm/labels/training.parquet"
MODELS        = ["cnn_tda"]

print(f"Train {len(SPLIT['train'])}개 / Val {len(SPLIT['val'])}개 / Test {len(SPLIT['test'])}개")

## 4. 리포지토리 설정

In [ ]:
import os, sys

REPO_DIR = "/content/ax_nilm"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhwang0803-glitch/ax_nilm {REPO_DIR} -q
    print("클론 완료")
else:
    !git -C {REPO_DIR} pull -q && echo "최신화 완료"

SRC_DIR     = f"{REPO_DIR}/nilm-engine/src"
SCRIPTS_DIR = f"{REPO_DIR}/nilm-engine/scripts"
CFG_DIR     = f"{REPO_DIR}/nilm-engine/config"

for d in [SRC_DIR, SCRIPTS_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)
print("경로 설정 완료")

## 5. 모듈 Import

In [ ]:
import yaml

from acquisition.gcs_loader import (
    list_channels_gcs, get_house_start_date_gcs,
    load_channel_data_gcs, get_appliance_name_gcs,
    GCSNILMDataset,
)
from acquisition.preprocessor import PowerScaler
from train_model import (
    build_model, masked_weighted_mse, evaluate, train_one_epoch,
    compute_pos_weight, _NILMDatasetWithTDA, APPLIANCE_LABELS,
)

with open(f"{CFG_DIR}/train.yaml")   as f: TRAIN_CFG   = yaml.safe_load(f)
with open(f"{CFG_DIR}/dataset.yaml") as f: DATASET_CFG = yaml.safe_load(f)

# EXP_ALL 설정 확인
exp_all_cfg = TRAIN_CFG["experiments"]["EXP_ALL"]
print(f"EXP_ALL config: week={exp_all_cfg['week']}  resume_from={exp_all_cfg['resume_from']}")
print("모든 모듈 import 완료")

## 6. Google Drive 마운트 (체크포인트 영구 저장)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

CKPT_DIR    = Path("/content/drive/MyDrive/nilm_run4/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/nilm_run4/results")
CACHE_DIR   = Path("/content/drive/MyDrive/nilm/cache")  # 기존 val/test 캐시 재사용
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"체크포인트 → {CKPT_DIR}")
print(f"결과       → {RESULTS_DIR}")
print(f"캐시       → {CACHE_DIR}")

## 7. 캐시 확인 (Val / Test)

> Train week1~4 캐시는 Run 3에서 이미 빌드됨 — 재사용.  
> Val(house_049) / Test(house_067)는 없으면 여기서 빌드.  
> TDA 캐시도 동일 cache_key로 자동 재사용.

In [ ]:
import gc

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context")
ss = DATASET_CFG["window"].get("steady_stride")

_ds_common = dict(
    gcs_fs=gcs,
    bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
    window_size=ws, stride=st,
    event_context=ec, steady_stride=ss,
    cache_dir=CACHE_DIR, denoise=True,
)

for split_name, houses, weeks in [
    ("train", SPLIT["train"], [1, 2, 3, 4]),
    ("val",   SPLIT["val"],   [None]),
    ("test",  SPLIT["test"],  [None]),
]:
    for w in weeks:
        label = f"{split_name}(week={w})"
        print(f"\n[{label}] 빌드 중...")
        base = GCSNILMDataset(
            houses, week=w,
            fit_scaler=(split_name == "train" and w == 1),
            **_ds_common,
        )
        print(f"  base: {len(base):,} windows")
        tda = _NILMDatasetWithTDA(base, cache_dir=CACHE_DIR)
        print(f"  TDA:  완료")
        del tda, base
        gc.collect()

print("\n✅ 전체 캐시 빌드 완료")

## 8. 함수 정의

In [ ]:
import json, time
import torch
import numpy as np
from torch.utils.data import DataLoader, ConcatDataset


def run_exp_gcs(exp_name: str, model_name: str,
                denoise: bool = True, tag: str = "", val_week=None) -> dict:
    """GCS 데이터로 단일 EXP/모델 학습. 결과를 Drive에 저장하고 metrics dict 반환."""
    exp_cfg = TRAIN_CFG["experiments"][exp_name]
    week = exp_cfg.get("week")
    ws   = DATASET_CFG["window"]["size"]
    st   = DATASET_CFG["window"]["stride"]
    ec   = DATASET_CFG["window"].get("event_context")
    ss   = DATASET_CFG["window"].get("steady_stride")
    bs   = TRAIN_CFG["training"]["batch_size"]
    ep   = TRAIN_CFG["training"]["epochs"]
    pat  = TRAIN_CFG["training"]["early_stopping_patience"]
    lr   = TRAIN_CFG["training"]["learning_rate"]
    wd   = TRAIN_CFG["optimizer"]["weight_decay"]
    lambda_mse = TRAIN_CFG["training"]["loss_weights"]["mse"]
    _suffix = f"_{tag}" if tag else ""
    _label  = f"{exp_name}/{model_name}" + (f"[{tag}]" if tag else "")

    print(f"\n{'='*58}")
    print(f"  {_label}  week={week}  denoise={denoise}")
    print(f"{'='*58}")

    # ── 1. 데이터셋 ──────────────────────────────────────────────────────────
    print("  [1/4] 데이터셋 로딩...")

    resume_exp = exp_cfg.get("resume_from")
    prev_scaler = None
    if resume_exp:
        scaler_path = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}_scaler.json"
        if scaler_path.exists():
            prev_scaler = PowerScaler.load(scaler_path)
            print(f"  └─ scaler 로드: mean={prev_scaler.mean:.2f}W  std={prev_scaler.std:.2f}W")

    _ds_common = dict(
        gcs_fs=gcs,
        bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st,
        event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR, denoise=denoise,
    )

    if week is None:
        # EXP_ALL: week1~4 캐시를 ConcatDataset으로 묶어 전체 데이터 단일 학습
        # week1로 scaler fit → 나머지 주차에 공유 (Run3와 동일 기준)
        _base_w1 = GCSNILMDataset(SPLIT["train"], week=1,
                                  fit_scaler=(prev_scaler is None),
                                  scaler=prev_scaler, **_ds_common)
        _shared_scaler = _base_w1.scaler
        _train_bases = [_base_w1] + [
            GCSNILMDataset(SPLIT["train"], week=w, scaler=_shared_scaler, **_ds_common)
            for w in [2, 3, 4]
        ]
        train_ds  = ConcatDataset([_NILMDatasetWithTDA(b, cache_dir=CACHE_DIR) for b in _train_bases])
        base_train = _base_w1  # scaler 저장용
        total_wins = sum(len(b) for b in _train_bases)
        print(f"  train(week1~4 concat): {total_wins:,} windows")
    else:
        if prev_scaler is not None:
            base_train = GCSNILMDataset(SPLIT["train"], scaler=prev_scaler, week=week, **_ds_common)
        else:
            base_train = GCSNILMDataset(SPLIT["train"], fit_scaler=True, week=week, **_ds_common)
        train_ds = _NILMDatasetWithTDA(base_train, cache_dir=CACHE_DIR)
        print(f"  train(week{week}): {len(train_ds):,} windows")

    base_val = GCSNILMDataset(SPLIT["val"], scaler=base_train.scaler, week=val_week, **_ds_common)
    val_ds   = _NILMDatasetWithTDA(base_val, cache_dir=CACHE_DIR)

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)
    print(f"  val: {len(val_ds):,} windows")

    from classifier.label_map import get_on_thresholds as _get_on_thr
    _raw_thr   = np.array(_get_on_thr(), dtype=np.float32)
    _scl       = base_train.scaler
    _fixed_thr = (_raw_thr - _scl.mean) / _scl.std if _scl is not None else None

    # ── 2. 모델 ──────────────────────────────────────────────────────────────
    print("  [2/4] 모델 초기화...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  device: {device}")
    model = build_model(model_name, ws).to(device)

    if resume_exp:
        prev_ckpt = CKPT_DIR / f"{resume_exp}_{model_name}{_suffix}.pt"
        if prev_ckpt.exists():
            _ckpt  = torch.load(prev_ckpt, map_location=device, weights_only=True)
            _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
            model.load_state_dict(_state)
            print(f"  └─ 모델 로드: {prev_ckpt.name}")

    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "min",
        factor=TRAIN_CFG["scheduler"]["factor"],
        patience=TRAIN_CFG["scheduler"]["patience"],
    )
    amp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    _app_index = {name: i for i, name in enumerate(APPLIANCE_LABELS)}
    _scale_cfg = TRAIN_CFG.get("appliance_loss_scale", {})
    appliance_scale = torch.ones(len(APPLIANCE_LABELS), device=device)
    for _name, _s in _scale_cfg.items():
        if _name in _app_index:
            appliance_scale[_app_index[_name]] = float(_s)
            print(f"  appliance_loss_scale [{_name}]: ×{_s}")

    pos_weight_max = float(TRAIN_CFG["training"].get("pos_weight_max", 20.0))
    print("  pos_weight 계산 중...")
    pos_weight = compute_pos_weight(train_dl, device, max_weight=pos_weight_max)
    for name, pw in zip(APPLIANCE_LABELS, pos_weight.cpu().tolist()):
        print(f"    {name}: {pw:.2f}")

    # ── 3. 학습 루프 ─────────────────────────────────────────────────────────
    print("  [3/4] 학습 시작...")
    best_score, best_cls_thresholds, best_vm, best_state, no_improve = \
        (-float("inf"), float("inf")), None, None, None, 0
    t0 = time.perf_counter()

    for epoch in range(1, ep + 1):
        loss = train_one_epoch(model, train_dl, optimizer, model_name, device,
                               amp_scaler=amp_scaler, pos_weight=pos_weight,
                               lambda_mse=lambda_mse, appliance_scale=appliance_scale)
        vm   = evaluate(model, val_dl, model_name, device, cls_thresholds=_fixed_thr)
        scheduler.step(vm["mae"])
        lr_now = optimizer.param_groups[0]["lr"]
        f1_cls_str = f"  f1_cls={vm['f1_cls']:.3f}" if vm.get("f1_cls") is not None else ""
        print(f"  ep {epoch:3d}/{ep}  loss={loss:.4f}  "
              f"val_mae={vm['mae']:.2f}W  f1={vm['f1']:.3f}{f1_cls_str}  lr={lr_now:.2e}")

        _f1_cls = vm.get("f1_cls") or 0.0
        _score  = (_f1_cls, -vm["mae"])
        if _score > best_score or best_state is None:
            best_score          = _score
            best_cls_thresholds = vm.get("best_cls_thresholds")
            best_vm             = vm
            best_state          = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve          = 0
        else:
            no_improve += 1
            if no_improve >= pat:
                print(f"  조기 종료 ({pat} epoch 개선 없음)")
                break

    elapsed = time.perf_counter() - t0

    # ── 4. 저장 ──────────────────────────────────────────────────────────────
    print("  [4/4] 체크포인트 & 메트릭 저장...")
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    torch.save({"model_state": model.state_dict(), "best_cls_thresholds": best_cls_thresholds},
               CKPT_DIR / f"{exp_name}_{model_name}{_suffix}.pt")
    if base_train.scaler is not None:
        base_train.scaler.save(CKPT_DIR / f"{exp_name}_{model_name}{_suffix}_scaler.json")

    fm = best_vm if best_vm is not None else vm
    fm.update({"exp": exp_name, "model": model_name, "week": week,
               "tag": tag, "denoise": denoise,
               "val_weeks": "all", "training_time_s": round(elapsed, 1),
               "final_lr": optimizer.param_groups[0]["lr"]})
    with open(RESULTS_DIR / f"{exp_name}_{model_name}{_suffix}_metrics.json", "w") as f:
        json.dump(fm, f, ensure_ascii=False, indent=2)

    f1_cls_str = f"  F1_cls={fm['f1_cls']:.3f}" if fm.get("f1_cls") is not None else ""
    print(f"  ✅  val MAE={fm['mae']:.2f}W  RMSE={fm['rmse']:.2f}W  "
          f"SAE={fm['sae']:.4f}  F1={fm['f1']:.3f}{f1_cls_str}  ({elapsed:.0f}s)")
    return fm


def write_results_md(exp_name: str) -> None:
    """EXP 결과 MD를 RESULTS_DIR에 생성."""
    from datetime import datetime

    exp_cfg  = TRAIN_CFG["experiments"][exp_name]
    week     = exp_cfg.get("week", "all")
    prev_exp = exp_cfg.get("resume_from")

    exp_ms  = {}
    prev_ms = {}
    for m in MODELS:
        p = RESULTS_DIR / f"{exp_name}_{m}_metrics.json"
        exp_ms[m] = json.load(open(p)) if p.exists() else None
        pp = RESULTS_DIR / f"{prev_exp}_{m}_metrics.json"
        prev_ms[m] = (json.load(open(pp)) if pp.exists() else None) if prev_exp else None

    lines = [
        f"# {exp_name} 실험 결과 (Run 4 — 전체 데이터 단일 학습)", "",
        "| 항목 | 값 |", "|------|-----|",
        "| 학습 데이터 | week1~4 ConcatDataset (8가구 전체 기간) |",
        f"| 이전 체크포인트 | {prev_exp or '처음부터'} |",
        "| Val 기간 | house_049 전체 기간 고정 |",
        f"| 기록 일시 | {datetime.now().strftime('%Y-%m-%d %H:%M')} |",
        "", "---", "",
        "## Val 성능", "",
        "| 모델 | MAE (W) | RMSE (W) | SAE | F1 | F1_cls | 학습시간 | final_lr |",
        "|------|---------|----------|-----|----|--------|---------|---------|",
    ]
    for m in MODELS:
        md = exp_ms.get(m)
        if md:
            f1_cls = f"{md['f1_cls']:.3f}" if md.get("f1_cls") is not None else "—"
            t_s    = md.get("training_time_s", "?")
            f_lr   = md.get("final_lr", "?")
            lines.append(f"| {m} | {md['mae']:.2f} | {md['rmse']:.2f} | {md['sae']:.4f} "
                         f"| {md['f1']:.3f} | {f1_cls} | {t_s}s | {f_lr:.2e} |")
        else:
            lines.append(f"| {m} | — | — | — | — | — | — | — |")

    lines += ["", "---", "", "## 메모", "",
              "> Run 3 최저 val MAE(EXP2=39.32W) 대비 비교 필요.",
              "> 개선 시: 누적 전략이 병목이었음. 미개선 시: 아키텍처 한계 의심.", ""]

    md_path = RESULTS_DIR / f"{exp_name}_results.md"
    md_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"MD 생성: {md_path}")


print("함수 정의 완료 — run_exp_gcs / write_results_md")

## 9. EXP_ALL 실행

> train 8가구 전체 기간 → scratch 학습 (epochs=25, lr=5e-4, λ_mse=5e-5)

In [ ]:

# ── GPU 투입 전 캐시 존재 확인 ─────────────────────────────────────────────
import hashlib
from features.tda import TDA_DIM

ws = DATASET_CFG["window"]["size"]
st = DATASET_CFG["window"]["stride"]
ec = DATASET_CFG["window"].get("event_context") or 0
ss = DATASET_CFG["window"].get("steady_stride") or 0

def _gcs_week_key(week):
    """GCSNILMDataset._week_key 와 동일한 공식"""
    return hashlib.md5(
        f"None|{week}|None|{BUCKET_PREFIX}|True".encode()
    ).hexdigest()[:8]

def _gcs_tda_key(houses, week):
    """GCSNILMDataset.cache_key 와 동일한 공식"""
    return hashlib.md5(
        f"{sorted(houses)}|None|{week}|None|{ws}|{st}|{BUCKET_PREFIX}|30|True".encode()
    ).hexdigest()[:12]

missing = []
for label, houses, week in [
    ("train_w1", SPLIT["train"], 1),
    ("train_w2", SPLIT["train"], 2),
    ("train_w3", SPLIT["train"], 3),
    ("train_w4", SPLIT["train"], 4),
    ("val",      SPLIT["val"],   None),
    ("test",     SPLIT["test"],  None),
]:
    wk  = _gcs_week_key(week)
    # base: 모든 house 파일이 있어야 함
    b_ok = all((CACHE_DIR / f"nilm_gcs_{h}_{wk}.npz").exists() for h in houses)
    # TDA: 통합 캐시 1개
    tda_key = _gcs_tda_key(houses, week)
    t_ok = (CACHE_DIR / f"tda_{tda_key}_ec{ec}_ss{ss}_d{TDA_DIM}.pt").exists()
    flag = "✅" if (b_ok and t_ok) else "❌"
    print(f"  {flag}  {label:10s}  base={'O' if b_ok else 'X'}  tda={'O' if t_ok else 'X'}")
    if not (b_ok and t_ok):
        missing.append(label)

if missing:
    raise RuntimeError(
        f"\n⛔ 캐시 누락: {missing}\n"
        "GPU 런타임을 끄고 CPU 런타임에서 셀 7(캐시 빌드)을 먼저 실행하세요."
    )
else:
    print("\n✅ 모든 캐시 확인 완료 — GPU 학습 진행 가능")

In [ ]:
result_all = run_exp_gcs("EXP_ALL", "cnn_tda", denoise=True)
write_results_md("EXP_ALL")

## 10. Test 평가 (oracle threshold)

> Val에서 grid-search로 찾은 최적 threshold 적용 — house_067 전체 기간.

In [ ]:
_model_name = "cnn_tda"
_exp        = "EXP_ALL"
ckpt_path   = CKPT_DIR / f"{_exp}_{_model_name}.pt"
scaler_path = CKPT_DIR / f"{_exp}_{_model_name}_scaler.json"

if not ckpt_path.exists():
    print(f"체크포인트 없음: {ckpt_path}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ws = DATASET_CFG["window"]["size"]
    st = DATASET_CFG["window"]["stride"]
    ec = DATASET_CFG["window"].get("event_context")
    ss = DATASET_CFG["window"].get("steady_stride")
    bs = TRAIN_CFG["training"]["batch_size"]

    scaler = PowerScaler.load(scaler_path) if scaler_path.exists() else None
    base_test = GCSNILMDataset(
        SPLIT["test"], scaler=scaler, week=None,
        gcs_fs=gcs, bucket_prefix=BUCKET_PREFIX, label_path=LABEL_PATH,
        window_size=ws, stride=st, event_context=ec, steady_stride=ss,
        cache_dir=CACHE_DIR, denoise=True,
    )
    test_ds = _NILMDatasetWithTDA(base_test, cache_dir=CACHE_DIR)
    test_dl = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=False)

    model = build_model(_model_name, ws).to(device)
    _ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    _state = _ckpt["model_state"] if isinstance(_ckpt, dict) and "model_state" in _ckpt else _ckpt
    _best_thr = _ckpt.get("best_cls_thresholds") if isinstance(_ckpt, dict) else None
    model.load_state_dict(_state)

    # oracle: val에서 찾은 최적 threshold
    _oracle_thr = np.array(_best_thr, dtype=np.float32) if _best_thr is not None else None

    print("=" * 58)
    print(f"  Test 평가 — {_exp} / oracle threshold")
    print("=" * 58)

    tm = evaluate(model, test_dl, _model_name, device, cls_thresholds=_oracle_thr)
    tm.update({"exp": _exp, "model": _model_name, "week": "all", "split": "test"})

    out_path = RESULTS_DIR / f"{_exp}_{_model_name}_test_oracle_metrics.json"
    with open(out_path, "w") as _f:
        json.dump(tm, _f, ensure_ascii=False, indent=2)

    print(f"  MAE={tm['mae']:.2f}W  RMSE={tm['rmse']:.2f}W  "
          f"SAE={tm['sae']:.4f}  F1={tm['f1']:.3f}  F1_cls={tm['f1_cls']:.3f}")
    print(f"  저장: {out_path.name}")

## 11. 결과 요약

In [ ]:
import json as _json

# Run 3 최고 성적 (EXP2 val / EXP4 test)
RUN3_BEST_VAL_MAE  = 39.32  # EXP2
RUN3_BEST_TEST_MAE = 44.14  # EXP4 oracle

val_p  = RESULTS_DIR / f"EXP_ALL_cnn_tda_metrics.json"
test_p = RESULTS_DIR / f"EXP_ALL_cnn_tda_test_oracle_metrics.json"

print("=" * 58)
print("  Run 4 (EXP_ALL) vs Run 3 비교")
print("=" * 58)

if val_p.exists():
    vm = _json.load(open(val_p))
    diff = vm['mae'] - RUN3_BEST_VAL_MAE
    trend = "↓ 개선" if diff < 0 else "↑ 악화"
    print(f"  Val  MAE: {vm['mae']:.2f}W  (Run3 best={RUN3_BEST_VAL_MAE}W  {trend} {abs(diff):.2f}W)")
    print(f"       RMSE={vm['rmse']:.2f}W  SAE={vm['sae']:.4f}  F1={vm['f1']:.3f}  F1_cls={vm.get('f1_cls', 0):.3f}")
    print(f"       학습시간={vm.get('training_time_s','?')}s  final_lr={vm.get('final_lr','?'):.2e}")

if test_p.exists():
    tm = _json.load(open(test_p))
    diff = tm['mae'] - RUN3_BEST_TEST_MAE
    trend = "↓ 개선" if diff < 0 else "↑ 악화"
    print(f"  Test MAE: {tm['mae']:.2f}W  (Run3 best={RUN3_BEST_TEST_MAE}W  {trend} {abs(diff):.2f}W)")
    print(f"       RMSE={tm['rmse']:.2f}W  SAE={tm['sae']:.4f}  F1={tm['f1']:.3f}  F1_cls={tm.get('f1_cls', 0):.3f}")

print("\n판단 기준:")
print("  Val MAE < 39.32W → 누적 전략이 병목이었음 → Run 5: 아키텍처 개선(B안) 진행")
print("  Val MAE ≈ Run3   → 누적 전략 무관 → λ_mse 재조정(C안) 또는 B안 검토")